# IAS Application

## 0. Install Packages

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install diffusers transformers accelerate peft
!pip install opencv-python pillow numpy tqdm
!pip install kornia pytorch-lightning timm einops
!pip install segmentation-models-pytorch@git+https://github.com/ragavsachdeva/segmentation_models.pytorch.git@0092ee4d6f851d89a4a401bb2dfa6187660b8dd3

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Path Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/Project-IAS'
DATA_ROOT = '/content/drive/MyDrive/AD_dataset_s_ROI'
DEFECT_TYPE = 'mounting_plate'

GENERATION_SRC = f'{PROJECT_ROOT}/generation_src'
PREPROCESSING_SRC = f'{PROJECT_ROOT}/pre-processing'

TRAIN_OUTPUT_DIR = '/content/sd_train_output'
SD_CKPT_DIR = f'{PROJECT_ROOT}/model/stable_diffusion'
CKPT_STEP = 600
CKPT_SUFFIX = '_application'
GEN_OUTPUT_DIR = '/content/synthesis_output_multi_seed'
RESTORED_DIR = '/content/restored_output_mounting_plate'

for path in [GENERATION_SRC, PREPROCESSING_SRC, f'{DATA_ROOT}/{DEFECT_TYPE}']:
    if not Path(path).exists():
        raise FileNotFoundError(f'경로를 찾을 수 없습니다: {path}')

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'DEFECT_TYPE  : {DEFECT_TYPE}')

## 3. Train Stable Diffusion

In [ ]:
%cd $GENERATION_SRC

!python train.py \
      --data_root $DATA_ROOT \
      --defect_type $DEFECT_TYPE \
      --pretrained_model_name_or_path SG161222/Realistic_Vision_V6.0_B1_noVAE \
      --vae_model_path stabilityai/sd-vae-ft-mse \
      --image_encoder_path laion/CLIP-ViT-H-14-laion2B-s32B-b79K \
      --output_dir $TRAIN_OUTPUT_DIR \
      --resolution 512 \
      --learning_rate 1e-5 \
      --weight_decay 1e-3 \
      --num_train_epochs 200 \
      --train_batch_size 4 \
      --dataloader_num_workers 2 \
      --save_steps 100 \
      --mixed_precision fp16 \
      --seed 42 \
      --num_visual_tokens 2 \
      --ckpt_suffix $CKPT_SUFFIX

## 4. Copy Checkpoints to Drive

In [ ]:
!mkdir -p {SD_CKPT_DIR}
!cp -r {TRAIN_OUTPUT_DIR}/checkpoint-{CKPT_STEP}/*{CKPT_SUFFIX}.bin {SD_CKPT_DIR}/

CONDITION_ENCODER_CKPT = f'{SD_CKPT_DIR}/condition_encoder-{CKPT_STEP}{CKPT_SUFFIX}.bin'
UNET_CKPT = f'{SD_CKPT_DIR}/unet-{CKPT_STEP}{CKPT_SUFFIX}.bin'

print(f'condition_encoder : {CONDITION_ENCODER_CKPT}')
print(f'unet              : {UNET_CKPT}')

## 5. Generate Synthetic Data (Multi-Seed)

In [ ]:
%cd $GENERATION_SRC

!python generation_multi_seed.py \
      --data_root $DATA_ROOT \
      --defect_type $DEFECT_TYPE \
      --pretrained_model_path SG161222/Realistic_Vision_V6.0_B1_noVAE \
      --vae_model_path stabilityai/sd-vae-ft-mse \
      --image_encoder_path laion/CLIP-ViT-H-14-laion2B-s32B-b79K \
      --condition_encoder_checkpoint $CONDITION_ENCODER_CKPT \
      --unet_checkpoint $UNET_CKPT \
      --output_dir $GEN_OUTPUT_DIR \
      --num_visual_tokens 2 \
      --num_generations 1 \
      --images_per_step 100 \
      --seed 42 \
      --num_inference_steps 50 \
      --quiet

## 6. Restore ROI to Original Coordinates

In [ ]:
%cd $PREPROCESSING_SRC

!python restore_generated_to_original.py \
      --defect_type $DEFECT_TYPE \
      --generated_dir $GEN_OUTPUT_DIR \
      --data_root $DATA_ROOT \
      --output_dir $RESTORED_DIR

## 7. Check Results

In [ ]:
import cv2
import matplotlib.pyplot as plt

images = sorted((Path(RESTORED_DIR) / 'images').glob('*.png'))
masks = sorted((Path(RESTORED_DIR) / 'masks').glob('*.png'))
print(f'복원 이미지 {len(images)}장 / 마스크 {len(masks)}장')

image = cv2.cvtColor(cv2.imread(str(images[0])), cv2.COLOR_BGR2RGB)
mask = cv2.imread(str(masks[0]), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image)
axes[0].set_title('restored image')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('restored mask')
axes[2].imshow(image)
axes[2].imshow(mask, cmap='jet', alpha=0.4)
axes[2].set_title('overlay')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()